# Nucleotide Transformer v2 test

In [3]:
import pandas as pd
import os

## Load data

In [4]:
base_path = "../../data/perphect-data"

couples_df = pd.read_csv(os.path.join(base_path, "public_data_set", "couples_df.csv"))
phages_df = pd.read_csv(os.path.join(base_path, "public_data_set", "phages_df.csv"))
bacteria_df = pd.read_csv(os.path.join(base_path, "public_data_set", "bacteria_df.csv"))
print('couples_df.shape = ', couples_df.shape)
print('phages_df.shape = ', phages_df.shape)
print('bacteria_df.shape = ', bacteria_df.shape)

couples_df.shape =  (4202, 4)
phages_df.shape =  (3459, 3)
bacteria_df.shape =  (94, 3)


In [5]:
phages_df.head()

,phage_id,phage_sequence,sequence_length
0,5326,TGCGGCCGCCCCATCCTGTACGGGTTTCCAAGTCGATCGGAGGGCA...,53124
1,6247,GGCTTTCGTGTGAGCCGTGATGTTTTCACGAATATGTGCCCCACCT...,74483
2,1976,GTGGGAATTTTTTTTTTGGGTTGCGCGGTGATCGCCGATGACGACG...,50781
3,430,ATGGCTTCGACTCAGACTCCAGCCGTCGGCAAGACCACGGCCATCG...,71565
4,431,TGCGGCTGAGCCATCGTGTACGGGTTTCCAAGTCCATCAGAGCCGG...,53396


## Get embedding using JAX

In [ ]:
import haiku as hk
import jax
import jax.numpy as jnp
from nucleotide_transformer.pretrained import get_pretrained_model

/home/carrillo/micromamba/envs/pbi-nt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Select a model from '500M_human_ref', '500M_1000G', '2B5_1000G', '2B5_multi_species', '50M_multi_species_v2', '100M_multi_species_v2', '250M_multi_species_v2', '500M_multi_species_v2', '1B_agro_nt'
model_name = '50M_multi_species_v2'

In [ ]:
max_length = int(max(map(len, phages_df["phage_sequence"][:2]))/6) + 1
max_length

12414

In [ ]:
# Get pretrained model
parameters, forward_fn, tokenizer, config = get_pretrained_model(
    model_name=model_name,
    embeddings_layers_to_save=(12,),
    attention_maps_to_save=((1, 4), (7, 16)),
    max_positions=32,
)
forward_fn = hk.transform(forward_fn)

Downloaded model's hyperparameters.
Downloaded model's weights...


In [ ]:
# Get data and tokenize it
sequences = [
    "ATTCCGAAATCGCTGACCGATCGTACGAAA",
    "ATTTCTCTCTCTCTCTGAGATCGATCGATCGATATCTCTCGAGCTAGC",
]
# sequences = list(phages_df["phage_sequence"])[:2]
tokens_ids = [b[1] for b in tokenizer.batch_tokenize(sequences)]
tokens_str = [b[0] for b in tokenizer.batch_tokenize(sequences)]
tokens = jnp.asarray(tokens_ids, dtype=jnp.int32)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [ ]:
# Initialize random key
random_key = jax.random.PRNGKey(0)

# Infer
outs = forward_fn.apply(parameters, random_key, tokens)
print(outs.keys())

dict_keys(['attention_map_layer_1_number_4', 'attention_map_layer_7_number_16', 'embeddings_12', 'logits'])


In [ ]:
embeddings = outs["embeddings_12"][:, 1:, :]  # removing CLS token
padding_mask = jnp.expand_dims(tokens[:, 1:] != tokenizer.pad_token_id, axis=-1)
masked_embeddings = embeddings * padding_mask  # multiply by 0 pad tokens embeddings
sequences_lengths = jnp.sum(padding_mask, axis=1)
mean_embeddings = jnp.sum(masked_embeddings, axis=1) / sequences_lengths
print(mean_embeddings.shape)

(2, 512)


In [ ]:
mean_embeddings

Array([[ 0.0307321 ,  0.28487912, -0.0241097 , ..., -0.00697614,
        -0.10268458, -0.20914476],
       [-0.17261767, -0.13952518, -0.12027154, ...,  0.15793578,
         0.15447846, -0.12072094]], dtype=float32)

## Embeddings using 🤗 transformers

In [42]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

In [43]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

'cuda:0'

In [44]:
# Import the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained("InstaDeepAI/nucleotide-transformer-v2-50m-multi-species", trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained("InstaDeepAI/nucleotide-transformer-v2-50m-multi-species", trust_remote_code=True)

In [45]:
# Choose the length to which the input sequences are padded. By default, the 
# model max length is chosen, but feel free to decrease it as the time taken to 
# obtain the embeddings increases significantly with it.
max_length = tokenizer.model_max_length
max_length

2048

In [46]:
# Create a dummy dna sequence and tokenize it
# sequences = ["ATTCCGATTCCGATTCCG", "ATTTCTCTCTCTCTCTGAGATCGATCGATCGAT"]
sequences = phages_df["phage_sequence"].iloc[:2]
print(f"{len(sequences)}, {len(sequences[0])}, {len(sequences[1])}")
tokens_ids = tokenizer.batch_encode_plus(sequences, return_tensors="pt", padding="max_length", max_length = max_length, truncation=True)["input_ids"].to(device)
tokens_ids.shape

2, 53124, 74483


torch.Size([2, 2048])

In [47]:
tokens_ids.shape

torch.Size([2, 2048])

In [48]:
model.to(device)

EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(4107, 512, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(2050, 512, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-11): 12 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((512,), eps=1e-12, elementwi

In [49]:
# Compute the embeddings
attention_mask = tokens_ids != tokenizer.pad_token_id
torch_outs = model(
    tokens_ids,
    attention_mask=attention_mask,
    encoder_attention_mask=attention_mask,
    output_hidden_states=True
)

In [52]:
# Compute sequences embeddings
embeddings = torch_outs['hidden_states'][-1].detach().cpu().numpy()
print(f"Embeddings shape: {embeddings.shape}")
# print(f"Embeddings per token: {embeddings}")

# Add embed dimension axis
attention_mask_unsq = torch.unsqueeze(attention_mask, dim=-1).cpu()

# Compute mean embeddings per sequence
mean_sequence_embeddings = torch.sum(attention_mask_unsq*embeddings, axis=-2)/torch.sum(attention_mask_unsq, axis=1)
print(f"Mean sequence embeddings: {mean_sequence_embeddings}")

Embeddings shape: (2, 2048, 512)
Mean sequence embeddings: tensor([[-0.2326,  0.3941,  0.3469,  ..., -0.1369, -0.1470, -0.2098],
        [-0.2374,  0.4609,  0.3259,  ..., -0.2885, -0.1596, -0.1963]])
